In [1]:
import os
from pathlib import Path
import ast

import pandas as pd

import spacy
from spacy import displacy
from spacy.tokens import Span

In [2]:
fp = Path("data/extracted_entities_30_06_2026.csv")
df = pd.read_csv(fp)
df = df.rename(columns={"geographic location" : "location"})

#, index_col=0)
# Entity values to python objects
entity_cols = ['person', 'location', 'organization']
df[entity_cols] = df[entity_cols].map(ast.literal_eval)
df

,filename,paragraph,person,location,organization
0,MMHCO01_000065498_mpeg21_a0001.json,EEISSTE 14 AM ER.,[],[],[]
1,MMHCO01_000065498_mpeg21_a0001.json,Zitting van Vrijdag 14 Januarij.,[],[],[]
2,MMHCO01_000065498_mpeg21_a0001.json,De beraadslaging over de begrooting voor de st...,[],[],"[{'start': 44, 'end': 60, 'text': 'staatsspoor..."
3,MMHCO01_000065498_mpeg21_a0001.json,De heer Ruijdecoper van Maarsseveen is van oor...,"[{'start': 0, 'end': 35, 'text': 'De heer Ruij...","[{'start': 78, 'end': 97, 'text': 'haven van H...",[]
4,MMHCO01_000065498_mpeg21_a0001.json,Zij ligt niet in de bedoeling der wet van 1860...,"[{'start': 0, 'end': 3, 'text': 'Zij', 'label'...","[{'start': 120, 'end': 140, 'text': 'Pruissisc...",[]
...,...,...,...,...,...
487,ddd_010484188_mpeg21_a0066.json,"Apz., no.",[],[],"[{'start': 0, 'end': 4, 'text': 'Apz.', 'label..."
488,ddd_010484188_mpeg21_a0066.json,"3, is aan den Oost-Indischen ambtenaar met ver...","[{'start': 64, 'end': 79, 'text': 'A. van Kerk...","[{'start': 53, 'end': 62, 'text': 'Nederland',...","[{'start': 110, 'end': 130, 'text': 'telegrafi..."
489,ddd_010902827_mpeg21_a0002.json,EERVOLLE vru meldingen.,[],[],"[{'start': 0, 'end': 8, 'text': 'EERVOLLE', 'l..."
490,ddd_010902827_mpeg21_a0002.json,"By afzonderlijke dagor- ' ders, zoo in IndiÃ« ...","[{'start': 105, 'end': 118, 'text': 'L. G. Fri...","[{'start': 53, 'end': 62, 'text': 'Nederland',...",[]


In [26]:
row_idx = 5

nlp = spacy.blank("nl")
row = df.iloc[row_idx]
text = row['paragraph']
doc = nlp(text)

raw_spans = row[entity_cols].explode().dropna().to_list()

converted_spans = []
for span in raw_spans:
    # doc.char_span returns a Span object (or None if it doesn't align with token boundaries)
    token_span = doc.char_span(span['start'], span['end'], label=span['label'][:3].upper())
    
    # It's good practice to filter out None values just in case of alignment issues
    if token_span is not None:
        converted_spans.append(token_span)

# --- THE MAGIC TRICK ---
# Instead of doc.spans["sc"], assign them directly to doc.ents
doc.ents = converted_spans

# Switch the style from "span" to "ent"
displacy.render(doc, style="ent")